<a href="https://colab.research.google.com/github/aleaurre/NeumoNet/blob/main/ucu_pneumonia_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCU Chest X-Ray Pneumonia Detection — Paso 1: baseline

**Tarea:** clasificar radiografías pediátricas en NORMAL (0) / PNEUMONIA (1).
**Métrica:** F1 (positivo = PNEUMONIA). **Submission:** `Id,Label` con label 0/1.

**Estrategia de este baseline**
- DenseNet-121 preentrenada (transfer learning).
- 5-fold estratificado → predicciones *out-of-fold* (OOF).
- **Umbral ajustado para maximizar F1** sobre las OOF (clave: F1 depende del umbral).
- Test = promedio de los 5 modelos del CV → se aplica el umbral.
- Manejo del desbalance con `pos_weight` en la BCE.

**Antes de correr — checklist:**
1. **Settings → Accelerator → GPU** (T4).
2. **Settings → Internet → ON** (para bajar los pesos preentrenados de ImageNet). Si la competencia no permite internet, ver la nota al final.

Corré las celdas de arriba hacia abajo. La celda del CV imprime el F1 por fold, el **F1 de CV**, y el F1 del baseline trivial (todo-pneumonia ≈ 0.77) para comparar.

In [ ]:
import os, glob, random, json
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models

class CFG:
    img_size = 224
    batch_size = 16     # más liviano en RAM/GPU
    epochs = 2          # solo para confirmar que corre
    folds = 2
    lr = 1e-4; weight_decay = 1e-4; seed = 42
    pretrained = True; num_workers = 2
    device = "cuda" if torch.cuda.is_available() else "cpu"
    weight_decay = 1e-4
    seed         = 42
    num_workers  = 2
    device       = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)
print("device:", CFG.device)

device: cpu


### 1. Localizar los datos y armar la tabla de entrenamiento

In [ ]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
import os, glob
import kagglehub
import pandas as pd, numpy as np
from sklearn.metrics import f1_score

# 1) descarga (queda cacheada; re-ejecutar es instantáneo) y define 'path'
path = kagglehub.competition_download('ucu-chest-x-ray-pneumonia-detection')
print("kagglehub path:", path)

# 2) detecta la carpeta que contiene train/ y test/
cand = [d for d in [path, *glob.glob(os.path.join(path, "*"))]
        if os.path.isdir(os.path.join(d, "train")) and os.path.isdir(os.path.join(d, "test"))]
assert cand, f"No encontré train/ y test/ dentro de {path}. Contenido: {os.listdir(path)}"
os.environ["DATA_DIR"] = cand[0]
print("DATA_DIR =", os.environ["DATA_DIR"], "->", os.listdir(os.environ["DATA_DIR"]))

# 3) arma la tabla de entrenamiento
def find_data_dir():
    for c in glob.glob("/kaggle/input/*") + [os.environ.get("DATA_DIR", ""), "./data"]:
        if c and os.path.isdir(f"{c}/train") and os.path.isdir(f"{c}/test"):
            return c
    raise FileNotFoundError("No encontré una carpeta con train/ y test/.")

IMG_EXTS = ("*.jpeg", "*.jpg", "*.png", "*.JPEG", "*.JPG", "*.PNG")
def list_images(folder):
    files = []
    for ext in IMG_EXTS:
        files += glob.glob(os.path.join(folder, ext))
    return sorted(files)

DATA_DIR = find_data_dir()
rows = []
for label_name, label in [("NORMAL", 0), ("PNEUMONIA", 1)]:
    for f in list_images(os.path.join(DATA_DIR, "train", label_name)):
        rows.append({"path": f, "label": label})
df = pd.DataFrame(rows).sample(frac=1, random_state=CFG.seed).reset_index(drop=True)

print("data dir:", DATA_DIR)
print(f"train: {len(df)} | NORMAL={int((df.label==0).sum())} PNEUMONIA={int((df.label==1).sum())}")
print(f"baseline trivial (todo-pneumonia) F1 = {f1_score(df.label, np.ones(len(df),int)):.4f}")

100%|██████████| 1.14G/1.14G [00:15<00:00, 78.9MB/s]

Extracting files...


kagglehub path: /root/.cache/kagglehub/competitions/ucu-chest-x-ray-pneumonia-detection
DATA_DIR = /root/.cache/kagglehub/competitions/ucu-chest-x-ray-pneumonia-detection -> ['test', 'sample_submission.csv', 'train']
data dir: /root/.cache/kagglehub/competitions/ucu-chest-x-ray-pneumonia-detection
train: 5232 | NORMAL=1349 PNEUMONIA=3883
baseline trivial (todo-pneumonia) F1 = 0.8520


### 2. Transforms, Dataset y modelo

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_transforms(train):
    if train:
        return T.Compose([
            T.Resize((CFG.img_size + 32, CFG.img_size + 32)),
            T.RandomResizedCrop(CFG.img_size, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
            T.RandomHorizontalFlip(0.5),
            T.RandomRotation(7),
            T.ColorJitter(brightness=0.1, contrast=0.1),
            T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return T.Compose([
        T.Resize((CFG.img_size, CFG.img_size)),
        T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

class XRayDataset(Dataset):
    def __init__(self, paths, labels=None, train=False):
        self.paths = list(paths); self.labels = labels; self.tf = build_transforms(train)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = self.tf(Image.open(self.paths[i]).convert("RGB"))
        if self.labels is not None:
            return x, torch.tensor(self.labels[i], dtype=torch.float32)
        return x

def build_model():
    weights = models.DenseNet121_Weights.DEFAULT if CFG.pretrained else None
    m = models.densenet121(weights=weights)
    m.classifier = nn.Linear(m.classifier.in_features, 1)   # 1 logit -> sigmoid
    return m

### 3. Funciones de entrenamiento, umbral e inferencia

In [ ]:
def train_one_fold(tr_df, va_df):
    dev = CFG.device
    n_pos = (tr_df.label == 1).sum(); n_neg = (tr_df.label == 0).sum()
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=dev)

    tr_dl = DataLoader(XRayDataset(tr_df.path.values, tr_df.label.values, train=True),
                       batch_size=CFG.batch_size, shuffle=True,
                       num_workers=CFG.num_workers, pin_memory=(dev=="cuda"))
    model = build_model().to(dev)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.epochs*max(len(tr_dl),1))
    use_amp = (dev == "cuda")
    try:    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (AttributeError, TypeError): scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    for epoch in range(CFG.epochs):
        model.train()
        for x, y in tr_dl:
            x, y = x.to(dev), y.to(dev).unsqueeze(1)
            opt.zero_grad()
            with torch.autocast(device_type="cuda", enabled=use_amp):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()

    model.eval(); probs = []
    with torch.no_grad():
        for x in DataLoader(XRayDataset(va_df.path.values, train=False),
                            batch_size=CFG.batch_size, num_workers=CFG.num_workers):
            probs.append(torch.sigmoid(model(x.to(dev))).squeeze(1).cpu().numpy())
    return model, np.concatenate(probs)

def best_threshold(y_true, y_prob):
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 91):
        f1 = f1_score(y_true, (y_prob >= t).astype(int))
        if f1 > best_f1: best_f1, best_t = f1, t
    return best_t, best_f1

@torch.no_grad()
def predict_test(models_list, test_paths):
    dev = CFG.device
    dl = DataLoader(XRayDataset(test_paths, train=False), batch_size=CFG.batch_size,
                    num_workers=CFG.num_workers)
    total = np.zeros(len(test_paths), dtype=np.float64)
    for model in models_list:
        model.eval(); parts = []
        for x in dl:
            parts.append(torch.sigmoid(model(x.to(dev))).squeeze(1).cpu().numpy())
        total += np.concatenate(parts)
    return total / len(models_list)

### 4. Validación cruzada (entrena los 5 folds)
Esta celda es la larga: en una T4 son ~25–45 min para 5×6 epochs.

In [ ]:
skf = StratifiedKFold(n_splits=CFG.folds, shuffle=True, random_state=CFG.seed)
oof_prob = np.zeros(len(df)); fold_models = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(df.path, df.label)):
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    va_df = df.iloc[va_idx].reset_index(drop=True)
    model, va_prob = train_one_fold(tr_df, va_df)
    oof_prob[va_idx] = va_prob; fold_models.append(model)
    t, f1 = best_threshold(va_df.label.values, va_prob)
    print(f"[fold {fold}] val F1={f1:.4f} @ t={t:.2f}")

THRESHOLD, CV_F1 = best_threshold(df.label.values, oof_prob)
print(f"\n>>> CV OOF F1 = {CV_F1:.4f}  @ threshold = {THRESHOLD:.3f}")

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 139MB/s]


[fold 0] val F1=0.9847 @ t=0.08
[fold 1] val F1=0.9838 @ t=0.07

>>> CV OOF F1 = 0.9840  @ threshold = 0.070


### 5. Inferencia en test y submission

In [ ]:
test_paths = list_images(os.path.join(DATA_DIR, "test"))
test_ids   = [os.path.splitext(os.path.basename(p))[0] for p in test_paths]
test_prob  = predict_test(fold_models, test_paths)
pred = dict(zip(test_ids, (test_prob >= THRESHOLD).astype(int)))

ssub = os.path.join(DATA_DIR, "sample_submission.csv")
if os.path.exists(ssub):
    sub = pd.read_csv(ssub); idc, labc = sub.columns[0], sub.columns[1]
    sub[labc] = sub[idc].astype(str).map(pred).fillna(0).astype(int)
else:
    sub = pd.DataFrame({"Id": test_ids, "Label": [pred[i] for i in test_ids]})

sub.to_csv("submission.csv", index=False)
pd.DataFrame({"path": df.path, "label": df.label, "oof_prob": oof_prob}).to_csv("oof.csv", index=False)
print("submission.csv:", sub.shape,
      "| pneumonia:", int((sub.iloc[:,1]==1).sum()), "normal:", int((sub.iloc[:,1]==0).sum()))
sub.head()

submission.csv: (624, 2) | pneumonia: 460 normal: 164


,Id,Label
0,test_0001,0
1,test_0002,0
2,test_0003,1
3,test_0004,1
4,test_0005,1


In [11]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=sub)

https://docs.google.com/spreadsheets/d/1f8KB-FLqJi44C_LIyVRbwukgmnwuyShlobz7TOVhYQA/edit#gid=0


### Cómo leer el resultado y qué sigue

- **El número que importa ahora es `CV OOF F1`.** Si es claramente > 0.77, ya le ganaste al baseline trivial. Para DenseNet-121 en estos datos esperá algo en torno a **0.95–0.97**.
- Subí el `submission.csv` a Kaggle y compará el **public LB** con tu CV. Si el LB queda muy por debajo del CV, hay sobreajuste del umbral o *leakage* del split — avisame y lo ajustamos.

**Reglas de decisión para el Paso 2 (mejoras, una por vez, midiendo contra este CV):**
1. **CLAHE** en el preproceso (contraste local) — barato, suele sumar.
2. **CBAM** sobre la DenseNet (atención canal+espacial) — el de mejor costo/beneficio.
3. **Focal loss** si la precisión de NORMAL es baja.
4. **TTA** (hflip) en inferencia — puntos casi gratis.
5. **Ensemble** con un segundo backbone (p. ej. ViT/EfficientNet preentrenado) promediando probabilidades.

Cada mejora se acepta solo si **sube el CV F1**; si no, se descarta. Decime el CV F1 que te dio y seguimos con el Paso 2.

---
**Nota si la competencia no permite Internet:** poné `CFG.pretrained = False` *no* es buena idea (cae mucho el F1). En su lugar, agregá como *Dataset* de entrada los pesos de `densenet121` de ImageNet y cargalos con `torch.hub.load_state_dict_from_file(...)` antes de reemplazar el classifier. Avisame y te paso ese bloque.